In [1]:
#  KÜTÜPHANE KURULUMLARI
# LangChain, HuggingFace, Vektör Veritabanı ve Arayüz kurulumu
!pip install -qU transformers accelerate langchain langchain-community langchain-huggingface
!pip install -qU pypdf faiss-cpu sentence-transformers gradio werkzeug

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.0/147.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.9/565.9 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

In [2]:
#  İÇE AKTARMALAR VE DOSYA GÜVENLİĞİ
import os
import torch
import gradio as gr
from werkzeug.utils import secure_filename
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

VECTOR_STORE = None
QA_CHAIN = None

def guvenli_dosya_kaydet(yuklenen_dosyalar):
    """
    İşletim sistemine sızma (Path Traversal) engellenir.
    Sadece PDF dosyalarının işleme girmesine izin verilir.
    """
    guvenli_yollar = []
    kayit_dizini = "./guvenli_pdf_deposu"
    os.makedirs(kayit_dizini, exist_ok=True)

    for dosya in yuklenen_dosyalar:
        orijinal_isim = os.path.basename(dosya.name)
        guvenli_isim = secure_filename(orijinal_isim)

        if not guvenli_isim.lower().endswith('.pdf'):
            continue

        hedef_yol = os.path.join(kayit_dizini, guvenli_isim)
        with open(dosya.name, 'rb') as f_in:
            with open(hedef_yol, 'wb') as f_out:
                f_out.write(f_in.read())

        guvenli_yollar.append(hedef_yol)

    return guvenli_yollar

/tmp/ipykernel_522/2789603164.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
#  T4 GPU İÇİN TAM UYUMLU VE HIZLI MODEL
import torch
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

print("Modeller maksimum hız ayarlarıyla GPU'ya yükleniyor...")

# 1. Embedding Modeli
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    model_kwargs={'device': 'cuda'}
)

# 2. Qwen2.5-3B Modeli
model_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16, # <-- T4 GPU'yu 2 kat hızlandırır
    attn_implementation="sdpa"
).to("cuda")

# 3. Boru Hattı (Performans Ayarları)
text_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,        # Uzun ve gereksiz metinleri engeller
    max_length=None,
    do_sample=False,           # <-- KRİTİK: İhtimal hesaplamayı kapatır, hızı anında artırır!
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
    device=0,
    return_full_text=False
)

yerel_llm = HuggingFacePipeline(pipeline=text_pipeline)
print(" Modeller T4 optimizasyonu ile başarıyla yüklendi!")

Modeller maksimum hız ayarlarıyla GPU'ya yükleniyor...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'eos_token_id', 'max_length', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


 Modeller T4 optimizasyonu ile başarıyla yüklendi!


In [5]:
#  RAG MOTORU VE CHATML PROMPT

def pdf_isleme_ve_vektor_olusturma(dosya_yollari):
    global VECTOR_STORE
    tum_sayfalar = []

    for yol in dosya_yollari:
        loader = PyPDFLoader(yol)
        sayfalar = loader.load()
        tum_sayfalar.extend(sayfalar)

    if not tum_sayfalar:
        return "Geçerli metin bulunamadı."

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=150,
        separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""]
    )

    parcalar = text_splitter.split_documents(tum_sayfalar)
    VECTOR_STORE = FAISS.from_documents(parcalar, embeddings)
    return f" {len(dosya_yollari)} adet PDF işlendi. Veri tabanı hazır!"

TURKCE_RAG_PROMPT = """<|im_start|>system
Sen kurumsal ve güvenilir bir yapay zeka asistanısın. Görevlerin:
1. Sadece sana verilen bağlam bilgilerini kullanarak kısa ve öz cevap ver.
2. Cevap bağlamda yoksa "Bu konuda bilgi bulunmamaktadır" de. Kendi bilgini uydurma.<|im_end|>
<|im_start|>user
Bağlam (Context):
{context}

Soru: {question}<|im_end|>
<|im_start|>assistant
"""

PROMPT = PromptTemplate(
    template=TURKCE_RAG_PROMPT,
    input_variables=["context", "question"]
)

def rag_soru_cevap(kullanici_sorusu):
    global VECTOR_STORE
    if VECTOR_STORE is None:
        return "Lütfen önce PDF dosyalarını yükleyin."

    #  k=4 yerine k=2 yapıldı.

    retriever = VECTOR_STORE.as_retriever(search_kwargs={"k": 2})
    ilgili_belgeler = retriever.invoke(kullanici_sorusu)

    context_text = "\n\n".join([doc.page_content for doc in ilgili_belgeler])
    final_prompt = PROMPT.format(context=context_text, question=kullanici_sorusu)

    try:
        cevap = yerel_llm.invoke(final_prompt)
        return cevap.strip()
    except Exception as e:
        return f"Bir hata oluştu: {str(e)}"

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import gradio as gr
import sqlite3
from datetime import datetime
import torch
from werkzeug.utils import secure_filename
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

VECTOR_STORE = None
QA_CHAIN = None

for _isim in ("model", "tokenizer", "text_pipeline", "yerel_llm", "embeddings"):
    if _isim in globals():
        del globals()[_isim]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    bos_bellek_gb = torch.cuda.mem_get_info()[0] / (1024 ** 3)
    print(f"GPU'da kullanılabilir boş bellek: {bos_bellek_gb:.2f} GiB")

print("Modeller maksimum hız ayarlarıyla yükleniyor...")

# 1. Embedding Modeli
# Not: Bu model küçüktür (~1 GB) ve GPU'da büyük bir kazanç sağlamaz.
# CPU'da tutmak, tüm GPU belleğini asıl büyük modele (Qwen2.5-3B) ayırmamızı sağlar.
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    model_kwargs={'device': 'cpu'}
)

# 2. Qwen2.5-3B Modeli
model_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16, # <-- T4 GPU'yu 2 kat hızlandırır
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
).to("cuda")

# 3. Boru Hattı (Performans Ayarları)
text_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,        # Uzun ve gereksiz metinleri engeller
    max_length=None,
    do_sample=False,           # <-- KRİTİK: İhtimal hesaplamayı kapatır, hızı anında artırır!
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
    device=0,
    return_full_text=False
)

yerel_llm = HuggingFacePipeline(pipeline=text_pipeline)
print(" Modeller T4 optimizasyonu ile başarıyla yüklendi!")

def guvenli_dosya_kaydet(yuklenen_dosyalar):
    """
    İşletim sistemine sızma (Path Traversal) engellenir.
    Sadece PDF dosyalarının işleme girmesine izin verilir.
    Birden fazla dosya aynı anda güvenli şekilde kaydedilir.
    """
    guvenli_yollar = []
    kayit_dizini = "./guvenli_pdf_deposu"
    os.makedirs(kayit_dizini, exist_ok=True)

    for dosya in yuklenen_dosyalar:
        orijinal_isim = os.path.basename(dosya.name)
        guvenli_isim = secure_filename(orijinal_isim)

        if not guvenli_isim.lower().endswith('.pdf'):
            continue

        hedef_yol = os.path.join(kayit_dizini, guvenli_isim)
        with open(dosya.name, 'rb') as f_in:
            with open(hedef_yol, 'wb') as f_out:
                f_out.write(f_in.read())

        guvenli_yollar.append(hedef_yol)

    return guvenli_yollar

# --- RAG MOTORU VE CHATML PROMPT ---
def pdf_isleme_ve_vektor_olusturma(dosya_yollari):
    global VECTOR_STORE
    tum_sayfalar = []
    basarisiz_dosyalar = []

    for yol in dosya_yollari:
        try:
            loader = PyPDFLoader(yol)
            sayfalar = loader.load()
            tum_sayfalar.extend(sayfalar)
        except Exception as e:
            basarisiz_dosyalar.append(os.path.basename(yol))
            print(f"PDF okuma hatası ({yol}): {e}")

    if not tum_sayfalar:
        return " Geçerli metin bulunamadı. "

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=150,
        separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""]
    )

    parcalar = text_splitter.split_documents(tum_sayfalar)
    VECTOR_STORE = FAISS.from_documents(parcalar, embeddings)

    basarili_sayisi = len(dosya_yollari) - len(basarisiz_dosyalar)
    isim_listesi = ", ".join([os.path.basename(y) for y in dosya_yollari if os.path.basename(y) not in basarisiz_dosyalar])
    mesaj = f" {basarili_sayisi} belge işlendi ve indekslendi: {isim_listesi}"
    if basarisiz_dosyalar:
        mesaj += f"\n İşlenemeyen dosyalar: {', '.join(basarisiz_dosyalar)}"
    return mesaj

TURKCE_RAG_PROMPT = """<|im_start|>system
Sen kurumsal ve güvenilir bir yapay zeka asistanısın. Görevlerin:
1. Sadece sana verilen bağlam bilgilerini kullanarak kısa ve öz cevap ver.
2. Cevap bağlamda yoksa "Bu konuda bilgi bulunmamaktadır" de. Kendi bilgini uydurma.<|im_end|>
<|im_start|>user
Bağlam (Context):
{context}

Soru: {question}<|im_end|>
<|im_start|>assistant
"""

PROMPT = PromptTemplate(
    template=TURKCE_RAG_PROMPT,
    input_variables=["context", "question"]
)

def rag_soru_cevap(kullanici_sorusu):
    global VECTOR_STORE
    if VECTOR_STORE is None:
        return "Lütfen önce PDF dosyalarını yükleyin."

    #  k=4 yerine k=2 yapıldı.

    retriever = VECTOR_STORE.as_retriever(search_kwargs={"k": 2})
    ilgili_belgeler = retriever.invoke(kullanici_sorusu)

    context_text = "\n\n".join([doc.page_content for doc in ilgili_belgeler])
    final_prompt = PROMPT.format(context=context_text, question=kullanici_sorusu)

    try:
        cevap = yerel_llm.invoke(final_prompt)
        return cevap.strip()
    except Exception as e:
        return f"Bir hata oluştu: {str(e)}"

# --- SQL VERİTABANI ALTYAPISI ---
def veritabani_kur():
    conn = sqlite3.connect('sohbet_gecmisi.db')
    c = conn.cursor()
    c.execute('''
        CREATE TABLE IF NOT EXISTS chat_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            tarih TEXT,
            kullanici_sorusu TEXT,
            bot_cevabi TEXT
        )
    ''')
    conn.commit()
    conn.close()

def log_kaydet(soru, cevap):
    conn = sqlite3.connect('sohbet_gecmisi.db')
    c = conn.cursor()
    zaman = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    c.execute("INSERT INTO chat_logs (tarih, kullanici_sorusu, bot_cevabi) VALUES (?, ?, ?)",
              (zaman, soru, cevap))
    conn.commit()
    conn.close()

def sql_verilerini_getir():
    """Arayüzdeki tabloyu beslemek için veritabanındaki kayıtları okur."""
    try:
        conn = sqlite3.connect('sohbet_gecmisi.db')
        c = conn.cursor()
        c.execute("SELECT id, tarih, kullanici_sorusu, bot_cevabi FROM chat_logs ORDER BY id DESC")
        veriler = c.fetchall()
        conn.close()
        return veriler
    except Exception as e:
        return [[f"Veritabanı Hatası: {e}", "", "", ""]]

veritabani_kur()


def arayuz_pdf_yukle(dosyalar):
    if not dosyalar:
        return " Lütfen en az bir PDF dosyası seçin."
    guvenli_dosyalar = guvenli_dosya_kaydet(dosyalar)
    if not guvenli_dosyalar:
        return " Sadece .pdf uzantılı dosyalar kabul edilir."
    durum_mesaji = pdf_isleme_ve_vektor_olusturma(guvenli_dosyalar)
    return durum_mesaji

def kullanici_mesaji_ekle(kullanici_mesaji, gecmis):
    if not kullanici_mesaji or not str(kullanici_mesaji).strip():
        return "", gecmis
    if gecmis is None:
        gecmis = []
    gecmis.append({"role": "user", "content": str(kullanici_mesaji)})
    return "", gecmis

def bot_yaniti_uret(gecmis):
    if not gecmis or gecmis[-1]["role"] != "user":
        return gecmis

    raw_content = gecmis[-1]["content"]

    if isinstance(raw_content, list):
        kullanici_mesaji = " ".join([
            str(item.get("text", item)) if isinstance(item, dict) else str(item)
            for item in raw_content
        ])
    elif isinstance(raw_content, tuple):
        kullanici_mesaji = str(raw_content[0])
    else:
        kullanici_mesaji = str(raw_content)

    kullanici_mesaji = kullanici_mesaji.strip()

    bot_mesaji = rag_soru_cevap(kullanici_mesaji)

    try:
        log_kaydet(kullanici_mesaji, bot_mesaji)
    except Exception as e:
        print(f"SQL Kayıt Hatası: {e}")

    gecmis.append({"role": "assistant", "content": bot_mesaji})
    return gecmis


RENK_ZEMIN = "#1F1E1D"
RENK_ZEMIN_KOYU = "#141312"
RENK_PANEL = RENK_ZEMIN
RENK_VURGU = "#6C9BD1"
RENK_VURGU_KOYU = "#5480AD"
BEYAZ = "#F5F4EF"
ACIK_METIN = "#B8B4AA"

SADE_CSS = f"""
.gradio-container {{
    background: {RENK_ZEMIN_KOYU} !important;
    font-family: 'Segoe UI', Arial, sans-serif !important;
}}

.gradio-container, .gradio-container p, .gradio-container span,
.gradio-container label, .gradio-container .prose {{
    color: {BEYAZ} !important;
}}

.gradio-container h1, .gradio-container h2, .gradio-container h3,
.gradio-container h4 {{
    color: {BEYAZ} !important;
}}

#baslik, #baslik * {{
    border: none !important;
    box-shadow: none !important;
}}
#baslik {{
    background: {RENK_ZEMIN};
    padding: 22px 26px;
    border-radius: 12px;
    margin-bottom: 18px;
    text-align: center;
}}

#dosya_yukle_ipucu, #dosya_yukle_ipucu * {{
    border: none !important;
    box-shadow: none !important;
    background: transparent !important;
}}

#kayitlar_ipucu, #kayitlar_ipucu * {{
    border: none !important;
    box-shadow: none !important;
    background: transparent !important;
}}

#baslik h1 {{
    color: {BEYAZ} !important;
    font-weight: 700 !important;
    font-size: 26px !important;
    text-align: center !important;
}}

#baslik p {{
    color: {BEYAZ} !important;
    font-size: 14px !important;
    text-align: center !important;
}}

/* Sekmeler */
button[role="tab"] {{
    color: {ACIK_METIN} !important;
    font-weight: 600 !important;
}}
button[role="tab"].selected {{
    color: {BEYAZ} !important;
    border-bottom: 3px solid {RENK_VURGU} !important;
}}

/* Kartlar / bloklar: ince, sade bir çerçeve ile görsel netlik korunur */
.gr-block, .block {{
    background: {RENK_ZEMIN} !important;
    border-radius: 10px !important;
    border: 1px solid rgba(255,255,255,0.08) !important;
    box-shadow: none !important;
}}

/* Chatbot/etiket ikonu (ör. "Sohbet" başlığı) düzgün hizalansın */
.label-wrap {{
    display: flex !important;
    align-items: center !important;
    gap: 6px !important;
}}
.label-wrap svg {{
    width: 15px !important;
    height: 15px !important;
    flex-shrink: 0 !important;
}}

/* Sohbet kutusu, diğer kutucuklarla aynı zemin rengini kullansın */
#yeni_sohbet_gecmisi_v5,
#yeni_sohbet_gecmisi_v5 .wrap,
#yeni_sohbet_gecmisi_v5 .bubble-wrap,
#yeni_sohbet_gecmisi_v5 > div {{
    background: {RENK_ZEMIN} !important;
}}

/* Köşe paneli (dosya yükleme + kayıtlar) */
#kose_paneli {{
    background: {RENK_PANEL} !important;
    border-radius: 12px !important;
    border: 1px solid rgba(255,255,255,0.08) !important;
    padding: 14px !important;
}}

/* "Örnek Sorular" bloğu: Sohbeti Sıfırla butonuyla aynı mavi çerçeve stili */
#ornek_blok {{
    border: 1px solid {RENK_VURGU} !important;
    border-radius: 8px !important;
    background: transparent !important;
    padding: 4px !important;
}}
#ornek_blok .block {{
    border: none !important;
    background: transparent !important;
}}

/* Girdi kutuları */
textarea, input[type="text"], input[type="search"] {{
    background: {RENK_ZEMIN_KOYU} !important;
    color: {BEYAZ} !important;
    border: 1px solid rgba(255,255,255,0.14) !important;
}}
textarea::placeholder, input::placeholder {{
    color: #8A8579 !important;
}}

/* Birincil butonlar */
button.primary, .primary {{
    background: {RENK_VURGU} !important;
    border: none !important;
    color: {BEYAZ} !important;
    font-weight: 700 !important;
}}
button.primary:hover, .primary:hover {{
    background: {RENK_VURGU_KOYU} !important;
}}

/* İkincil butonlar */
button.secondary, .secondary {{
    background: transparent !important;
    border: 1px solid {RENK_VURGU} !important;
    color: {RENK_VURGU} !important;
    font-weight: 600 !important;
}}

#durum_kutusu textarea {{
    border-left: 3px solid {RENK_VURGU} !important;
}}

/* Dataframe (SQL tablosu): gövde metniyle aynı yazı stili, ince çerçeve */
.gradio-container .table-wrap {{
    border: 1px solid rgba(255,255,255,0.08) !important;
    border-radius: 8px !important;
}}
table, th, td {{
    color: {BEYAZ} !important;
    background: {RENK_ZEMIN} !important;
    font-family: 'Segoe UI', Arial, sans-serif !important;
    font-weight: 400 !important;
    font-size: 14px !important;
}}
thead th {{
    font-weight: 600 !important;
    color: {BEYAZ} !important;
    background: {RENK_ZEMIN} !important;
}}
"""

SADE_TEMA = gr.themes.Default(
    primary_hue=gr.themes.colors.blue,
    secondary_hue=gr.themes.colors.stone,
    neutral_hue=gr.themes.colors.stone,
    font=[gr.themes.GoogleFont("Inter"), "Segoe UI", "Arial", "sans-serif"],
).set(
    body_background_fill=RENK_ZEMIN_KOYU,
    body_text_color=BEYAZ,
    block_background_fill=RENK_ZEMIN,
    block_label_text_color=BEYAZ,
    button_primary_background_fill=RENK_VURGU,
    button_primary_background_fill_hover=RENK_VURGU_KOYU,
    button_primary_text_color=BEYAZ,
    block_title_text_weight="600",
)

with gr.Blocks(theme=SADE_TEMA, css=SADE_CSS, title="Local RAG Assistant") as demo:

    with gr.Column(elem_id="baslik"):
        gr.Markdown("# Local RAG Assistant")

    with gr.Row():
        # ANA ALAN: SOHBET
        with gr.Column(scale=3):
            sohbet_ekrani = gr.Chatbot(
                elem_id="yeni_sohbet_gecmisi_v5",
                height=520,
                label="Sohbet",
            )

            with gr.Row():
                mesaj_kutusu = gr.Textbox(
                    placeholder="Bir soru sorun...",
                    show_label=False, container=False, scale=8
                )
                gonder_butonu = gr.Button("Gönder", variant="primary", scale=1)

            with gr.Row():
                temizle_butonu = gr.ClearButton(
                    [mesaj_kutusu, sohbet_ekrani],
                    value="Sohbeti Sıfırla",
                    size="sm",
                    variant="secondary",
                )

            with gr.Column(elem_id="ornek_blok"):
                gr.Examples(
                    examples=[
                        "Bu belgenin ana konusunu bir paragrafta özetler misin?",
                        "Metindeki en önemli 3 bulgu veya veri nedir?",
                        "Belgede önerilen çözümler veya tavsiyeler neler?"
                    ],
                    inputs=mesaj_kutusu
                )

            mesaj_kutusu.submit(
                kullanici_mesaji_ekle, [mesaj_kutusu, sohbet_ekrani], [mesaj_kutusu, sohbet_ekrani], queue=False
            ).then(bot_yaniti_uret, sohbet_ekrani, sohbet_ekrani)

            gonder_butonu.click(
                kullanici_mesaji_ekle, [mesaj_kutusu, sohbet_ekrani], [mesaj_kutusu, sohbet_ekrani], queue=False
            ).then(bot_yaniti_uret, sohbet_ekrani, sohbet_ekrani)

        # KÖŞE PANELİ: DOSYA YÜKLEME + SİSTEM KAYITLARI (kompakt, sağ üstte)
        with gr.Column(scale=1, elem_id="kose_paneli"):
            with gr.Tabs():
                with gr.TabItem(" Dosya Yükle "):
                    gr.Markdown(
                        f"<span style='color:{ACIK_METIN}; font-size: 12px;'>PDF belgelerinizi buradan yükleyin. Yüklenen dosyalar analiz edilip indekslenir, ardından içerikleri hakkında soru sorabilirsiniz.</span>",
                        elem_id="dosya_yukle_ipucu",
                    )
                    dosya_yukleyici = gr.File(
                        label="PDF Dosyaları",
                        file_count="multiple",
                        file_types=[".pdf"],
                    )
                    yukleme_butonu = gr.Button( "Belgeleri Hazırla ", variant="primary", size="sm")
                    durum_ciktisi = gr.Textbox(
                        label="Sistem Durumu",
                        interactive=False,
                        placeholder=" Henüz belge yüklenmedi. ",
                        elem_id="durum_kutusu",
                        lines=3,
                    )
                    yukleme_butonu.click(fn=arayuz_pdf_yukle, inputs=[dosya_yukleyici], outputs=[durum_ciktisi])

                with gr.TabItem(" Kayıtlar "):
                    gr.Markdown(
                        f"<span style='color:{ACIK_METIN}; font-size: 12px;'>Geçmiş sorularınızı ve aldığınız yanıtları buradan takip edin.</span>",
                        elem_id="kayitlar_ipucu",
                    )
                    log_tablosu = gr.Dataframe(
                        headers=["ID", "Tarih", "Soru", "Cevap"],
                        datatype=["number", "str", "str", "str"],
                        interactive=False,
                        wrap=True,
                    )
                    yenile_butonu = gr.Button("Kayıtları Yenile", variant="secondary", size="sm")
                    yenile_butonu.click(fn=sql_verilerini_getir, inputs=[], outputs=[log_tablosu])

demo.launch(debug=True, share=True)

GPU'da kullanılabilir boş bellek: 14.46 GiB
Modeller maksimum hız ayarlarıyla yükleniyor...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

 Modeller T4 optimizasyonu ile başarıyla yüklendi!


/tmp/ipykernel_522/2901386594.py:437: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=SADE_TEMA, css=SADE_CSS, title="Local RAG Assistant") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://82466dd6c52f9b06ce.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
